# Step 1 — Install and import:




In [5]:
!pip install -q transformers accelerate

import numpy as np
from transformers import pipeline

# Step 2 — Knowledge base:



In [8]:
DOCUMENTS = [
    "France Q3 sales: 1.2M EUR. Top products: WidgetA, WidgetB.",
    "Germany Q3 sales: 1.8M EUR. Top products: WidgetB, WidgetC.",
    "Recall notice: WidgetB recalled in EU as of Oct 2025. WidgetC under review.",
    "EU refund policy: 30 days return plus 14-day cooling-off period.",
    "US refund policy: 14 days return, no cooling-off period.",
]

# Step 3 — Embedding and similarity:



In [9]:
def embed(text):
    vocab = set(" ".join(DOCUMENTS).lower().split())
    vec = np.zeros(len(vocab))
    words = text.lower().split()
    for i, w in enumerate(sorted(vocab)):
        vec[i] = words.count(w)
    return vec

def cosine(a, b):
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    return float(np.dot(a, b) / denom) if denom else 0.0

# Step 4 — Retriever:



In [10]:
def retrieve(query, top_k=2):
    q_vec = embed(query)
    scored = [(cosine(q_vec, embed(doc)), doc) for doc in DOCUMENTS]
    scored.sort(reverse=True)
    return [doc for _, doc in scored[:top_k]]

# Step 5 — Load the LLM (slow first time):



In [14]:
llm = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.2",
    device_map="auto",
    torch_dtype="auto",
)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

# Step 6 — Generate function:



In [17]:
def generate(prompt):
    out = llm(prompt, max_new_tokens=200, do_sample=False)
    return out[0]["generated_text"][len(prompt):].strip()

# Step 7 — RAG pipeline:



In [18]:
def regular_rag(query, top_k=2):
    docs = retrieve(query, top_k=top_k)
    context = "\n".join(f"- {d}" for d in docs)
    prompt = (
        "Answer the question using only the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )
    return generate(prompt), docs

# Step 8 — Run:



In [19]:
query = "Compare France vs Germany sales and flag any recalls"
answer, docs = regular_rag(query)

print("Retrieved docs:")
for d in docs:
    print(" -", d)
print("\nAnswer:")
print(answer)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieved docs:
 - Germany Q3 sales: 1.8M EUR. Top products: WidgetB, WidgetC.
 - France Q3 sales: 1.2M EUR. Top products: WidgetA, WidgetB.

Answer:
In Q3, Germany had higher sales than France with a total of 1.8M EUR compared to France's 1.2M EUR. The top-selling products in both countries were WidgetB. No product recalls were mentioned in the context.
